## Customers Table - PySpark Cleaning

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, to_date, trim
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("CustomersCleaning").getOrCreate()

df_customers = spark.read.csv("../Coursework_dataset/customers.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("../Coursework_dataset/orders.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 18:17:19 WARN Utils: Your hostname, saag, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface wlp1s0)
26/04/14 18:17:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 18:17:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 18:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Preview the customers table

In [2]:
df_customers.show(5)

+-----------+---------------+--------------------+---------+-----------------+----------+----------------+-----------------+---------+
|customer_id|  customer_name|               email|     city|   state_province|   country|customer_segment|registration_date|age_group|
+-----------+---------------+--------------------+---------+-----------------+----------+----------------+-----------------+---------+
|cust_000001|Prakash Pradhan|prakash.pradhan28...|Kathmandu| Bagmati Province|     Nepal|          Retail|       2021-01-10|    18-25|
|cust_000002|     Salim Khan|salim.khan693@gma...| Rajshahi|Rajshahi Division|Bangladesh|       Wholesale|       2020-08-03|    36-50|
|cust_000003|    Anita Patel|anita.patel96@yah...|Bangalore|        Karnataka|     India|          Retail|       2019-02-05|    36-50|
|cust_000004|    Puspa Baral|puspa.baral28@yah...|  Hetauda| Bagmati Province|     Nepal|          Retail|       2023-10-21|    36-50|
|cust_000005|Radha Manandhar|radha.manandhar60...|  Bir

### Check how many values each column has (non-null count)

In [3]:
non_null_counts = df_customers.select([
    count(when(col(c).isNotNull(), c)).alias(c) for c in df_customers.columns
])
non_null_df = non_null_counts.toPandas().T.reset_index()
non_null_df.columns = ['Column Name', 'Non-Null Count']
non_null_df

,Column Name,Non-Null Count
0,customer_id,120000
1,customer_name,120000
2,email,119200
3,city,120000
4,state_province,120000
5,country,120000
6,customer_segment,120000
7,registration_date,120000
8,age_group,119700


### Check which columns have null values

In [4]:
null_counts = df_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns
])
null_df = null_counts.toPandas().T.reset_index()
null_df.columns = ['Column Name', 'Null Count']
null_df

,Column Name,Null Count
0,customer_id,0
1,customer_name,0
2,email,800
3,city,0
4,state_province,0
5,country,0
6,customer_segment,0
7,registration_date,0
8,age_group,300


### Fix 1: Fill null emails with a placeholder
Customers with no email get "unknown@domain.com" so the column has no blanks.

In [5]:
df_customers = df_customers.withColumn(
    "email",
    when(col("email").isNull(), lit("unknown@domain.com"))
    .otherwise(col("email"))
)

print("Null emails remaining:", df_customers.filter(col("email").isNull()).count())

Null emails remaining: 0


### Check 2: Find emails with invalid format
A valid email looks like `name@domain.com`. We flag rows that do not match this pattern.

In [6]:
# Emails that do not match the standard format
email_pattern = r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'

invalid_emails = df_customers.filter(
    ~col("email").rlike(email_pattern)
)

print("Rows with invalid email format:", invalid_emails.count())
invalid_emails.select("customer_id", "email").show(10, truncate=False)

Rows with invalid email format: 175
+-----------+-------------------------------+
|customer_id|email                          |
+-----------+-------------------------------+
|cust_000834|lalitha.de silva5@outlook.com  |
|cust_002528|hasaranga.de silva532@yahoo.com|
|cust_003624|sanath.de silva340@yahoo.com   |
|cust_003693|dinesh.de silva458@outlook.com |
|cust_004657|subashini.de silva885@yahoo.com|
|cust_004906|pathum.de silva713@outlook.com |
|cust_006257|nilmini.de silva634@hotmail.com|
|cust_007229|inoka.de silva443@outlook.com  |
|cust_008035|bhanuka.de silva716@outlook.com|
|cust_008432|wanindu.de silva79@outlook.com |
+-----------+-------------------------------+
only showing top 10 rows


### Fix 3: Fill null age_group with -1
Using -1 instead of a label like 'unknown' so that it does not affect the distribution of real age groups.

In [7]:
df_customers = df_customers.withColumn(
    "age_group",
    when(col("age_group").isNull(), lit("-1"))
    .otherwise(col("age_group"))
)

print("Null age_group remaining:", df_customers.filter(col("age_group").isNull()).count())

Null age_group remaining: 0


### Fix 4: Remove duplicate customer_id rows
Keep the first occurrence of each customer_id and drop the rest.

In [8]:
before = df_customers.count()
df_customers = df_customers.dropDuplicates(["customer_id"])
after = df_customers.count()

print(f"Rows before dedup: {before}, after: {after}, removed: {before - after}")

Rows before dedup: 120000, after: 120000, removed: 0


### Check 5: Find duplicate emails
Multiple customers sharing the same email causes identity ambiguity. We report how many there are.

In [9]:
# Find emails that appear more than once (exclude the placeholder)
dup_emails = df_customers.filter(col("email") != "unknown@domain.com") \
    .groupBy("email").count() \
    .filter(col("count") > 1)

print("Emails shared by more than one customer:", dup_emails.count())
dup_emails.show(10, truncate=False)

Emails shared by more than one customer: 153
+------------------------------+-----+
|email                         |count|
+------------------------------+-----+
|mukesh.magar499@outlook.com   |2    |
|dinesh.kulkarni194@hotmail.com|2    |
|sita.tuladhar902@gmail.com    |2    |
|janaki.magar529@yahoo.com     |2    |
|hem.dhakal814@hotmail.com     |2    |
|chandra.poudel731@outlook.com |2    |
|sapana.rijal859@gmail.com     |2    |
|nisha.karki922@hotmail.com    |2    |
|ranjana.mishra601@hotmail.com |2    |
|mahesh.gyawali257@outlook.com |2    |
+------------------------------+-----+
only showing top 10 rows


### Check 6: Find customers assigned more than one age_group
Each customer should have exactly one age group. More than one is a data error.

In [10]:
# Count distinct age_groups per customer
multi_age = df_customers.groupBy("customer_id") \
    .agg(F.countDistinct("age_group").alias("distinct_age_groups")) \
    .filter(col("distinct_age_groups") > 1)

print("Customers with multiple age_group values:", multi_age.count())
multi_age.show(10)

Customers with multiple age_group values: 0
+-----------+-------------------+
|customer_id|distinct_age_groups|
+-----------+-------------------+
+-----------+-------------------+



### Fix 7: Add dummy rows for orphan customer_ids in orders
Some orders reference a customer_id that does not exist in the customers table. 
These are called orphan records. We create dummy parent rows so referential integrity is maintained.

In [11]:
# Find order customer_ids that are not in the customers table
orphan_customer_ids = df_orders.select("customer_id") \
    .join(df_customers.select("customer_id"), on="customer_id", how="left_anti")

print("Orphan customer_ids found in orders:", orphan_customer_ids.count())
orphan_customer_ids.show(10)

Orphan customer_ids found in orders: 1202
+-----------+
|customer_id|
+-----------+
|cust_120339|
|cust_120330|
|cust_120204|
|cust_121003|
|cust_120863|
|cust_120244|
|cust_120396|
|cust_120071|
|cust_121115|
|cust_120087|
+-----------+
only showing top 10 rows


In [12]:
# Build dummy rows for each missing customer_id
# All other fields get a placeholder value so the row is valid
dummy_customers = orphan_customer_ids.distinct().select(
    col("customer_id"),
    lit("Unknown").alias("customer_name"),
    lit("unknown@domain.com").alias("email"),
    lit("Unknown").alias("city"),
    lit("Unknown").alias("state_province"),
    lit("Unknown").alias("country"),
    lit("Unknown").alias("customer_segment"),
    lit("1900-01-01").alias("registration_date"),
    lit("-1").alias("age_group")
)

# Add the dummy rows to the customers table
df_customers = df_customers.unionByName(dummy_customers)

print("Total customers after adding dummies:", df_customers.count())

Total customers after adding dummies: 121200


### Final check: Verify null counts after all cleaning

In [13]:
final_nulls = df_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns
])
final_null_df = final_nulls.toPandas().T.reset_index()
final_null_df.columns = ['Column Name', 'Null Count']
final_null_df

,Column Name,Null Count
0,customer_id,0
1,customer_name,0
2,email,0
3,city,0
4,state_province,0
5,country,0
6,customer_segment,0
7,registration_date,0
8,age_group,0


### Export the cleaned customers table

In [14]:
df_customers.coalesce(1).write.csv("../cleaned_dataset/customers_cleaned.csv", header=True, mode="overwrite")
print("Customers table exported successfully to 'cleaned_dataset/customers_cleaned.csv'")

Customers table exported successfully to 'cleaned_dataset/customers_cleaned.csv'
